In [ ]:
import os
os.environ["GROQ_API_KEY"] = ""

## Install libraries

In [ ]:
!pip install -q youtube-transcript-api langchain-community langchain-groq \
               faiss-cpu tiktoken python-dotenv

In [ ]:
!pip install -U langchain-text-splitters

In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate


## Step 1a - Indexing (Document Ingestion)

In [ ]:
video_id = "OWW1R50YiPE"

try:
    transcript_list = YouTubeTranscriptApi().fetch(video_id)


    transcript = " ".join(chunk.text for chunk in transcript_list)

    import re
    transcript = re.sub(r">>", "", transcript)

    print(transcript)

except TranscriptsDisabled:
    print("No captions available for this video.")

What is your problem?  Oh, I don't have a problem. You should probably wash your hands.  I just witnessed an altercation between you and Georgie Cooper.  Altercation?  A confrontation? A tussle. Anyway, I'd just like to gather more information for sociological purposes.  Oh, I heard about you. You're that smart kid.  I'd like to be humble in this moment, but yes, that's me. Sheldon Cooper at your service.  So Georgie is your brother?  Correct.  And you're trying to protect him?  Incorrect. I'm just curious what he did to incur your wrath. Also, kudos on the handwashing.  Your brother is a punk.  I'm not familiar with that terminology.  He tried to hit on my girlfriend.  Interesting. So, he openly pursued your mate, and to reassert dominance, you threatened him with physical violence.  Hell yeah, I did. I understand that. I often intimidate people with my intelligence.  Well, one of us scared him.  Good news. I just spoke to Tommy Clarkson.  What? Are you crazy?  Nope. Mom had me tested

## Step 1b - Indexing (Text Splitting)

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000 , chunk_overlap = 200)
chunks = splitter.create_documents([transcript])

In [ ]:
len(chunks)

33

In [ ]:
chunks[32]

Document(metadata={}, page_content="Jerome Freriedman, and Richard Taylor for the discovery of quarks. A primary feature of quarks is that they're always bonded together. But in that moment, I felt like a nutrino destined to be alone forever.  Leonard dear, you should be in bed. [Music]  Want to say to say it. [Music] [Applause] Yes, we will. Yes, we will.  Howard, turn off that preca game and go to sleep.  Someday we'll be together.  Yes, we will. Yes, we will.  I know.  Someday  we'll be together. Thankfully, I was wrong.")

## Step 1c & 1d - Indexing (Embedding Generation and Storing in Vector Store)

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vector_store = FAISS.from_documents(chunks, embeddings)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
vector_store.index_to_docstore_id

{0: '87708458-73c7-42ce-a94e-cb0b51207c70',
 1: 'ed04cc89-2319-4ae7-a8fc-0ee130e26555',
 2: 'c2237cb4-41a4-48c2-8862-fa917cd1df2e',
 3: '3077f08b-56b7-4c45-bc6b-33cf70430cc3',
 4: '80afa31a-c920-40d7-bbb1-94f312a3696e',
 5: 'ae5a42f0-e946-4136-b979-508c82e11448',
 6: '123ab742-3f22-41d9-9506-7f77014bff3a',
 7: '328901a4-a803-4493-966d-16907a00f0fd',
 8: '2611f43d-8a2c-4575-9579-fa7d8da8b8ce',
 9: '76a43613-f628-4b51-a92c-ca82fdb92709',
 10: '582babb5-bf03-4d4b-9305-252b8a74b480',
 11: '55f9a4db-e625-465b-bd0a-7a3b7ee3516f',
 12: '0ce4df1e-fcd6-4934-9a73-1a643f3f1693',
 13: '084e73d7-597e-4f5a-930f-6c8b974f8086',
 14: '761d0708-0c74-409e-99b9-5d36548cfa73',
 15: 'aed6e1e5-1a02-4bca-847d-5544c776baee',
 16: '582e7c8c-e5e4-49be-8b62-90384e21e053',
 17: '4dc0fe98-f865-4914-ba59-2f35af6d8e0e',
 18: '82291124-21eb-4e29-83a8-c802bbeb80d3',
 19: '3c6c5431-6027-4520-990f-88a4d7ae9fd4',
 20: '804a7d2c-3a3f-4a42-b2c3-4d68e19d7f02',
 21: 'a853c86a-149c-4517-ba86-e05e3bd880b5',
 22: 'e0030c0a-f408-

In [ ]:
vector_store.get_by_ids(['01c6ffb6-9bd3-4c29-8856-e96e45a05013'])

[]

## Step 2 - Retrieval

In [ ]:
retriever = vector_store.as_retriever(search_type = "similarity" , search_kwargs={"k":4})

In [ ]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7f02d8d19730>, search_kwargs={'k': 4})

## Step 3 - Augmentation

In [ ]:
llm = ChatGroq(
    model_name="llama-3.1-8b-instant",
    temperature=0.2
)

In [ ]:
prompt = PromptTemplate(
    template = """
     you are a helpful assistant.
     Answer ONLY from the provided transcript context.
     If the context is insufficient, just say you don't know.

     {context}
     Question: {question}

     """,
    input_variables = ['context','question']
)

In [ ]:
question = "Do missy stole makeup?"
retrived_docs = retriever.invoke(question)

In [ ]:
retrived_docs

[Document(id='c2237cb4-41a4-48c2-8862-fa917cd1df2e', metadata={}, page_content="you look like Madonna?  Thank you. I was actually trying to look like you.  Thank you.  Why would you give her makeup without talking to me first?  I didn't give her any makeup. Well, then one of you is lying.  She is.  She is.  Time to come clean, little lady. I ain't taking the hit on this one.  I took the makeup from me, mouse bag.  Oh, Missy, you are in a world of trouble.  Calm down. It's not that big a deal.  It is so a big deal. She stole from you and then she lied about it.  I know, but come on. She's only 10. She's going to do way stupider stuff when she's older.  Guaranteed.  I don't care. She is my daughter and you will be punished.  Sorry, kid.  I wish I was your daughter.  Is that so? Well, guess what? You're sleeping here tonight because I don't feel like being around either one of you.  Great.  Yeah, great. There was a point where the doctors didn't know if you were going to make it and your 

In [ ]:
context_text = "\n\n".join(doc.page_content for doc in retrived_docs)

In [ ]:
context_text

"you look like Madonna?  Thank you. I was actually trying to look like you.  Thank you.  Why would you give her makeup without talking to me first?  I didn't give her any makeup. Well, then one of you is lying.  She is.  She is.  Time to come clean, little lady. I ain't taking the hit on this one.  I took the makeup from me, mouse bag.  Oh, Missy, you are in a world of trouble.  Calm down. It's not that big a deal.  It is so a big deal. She stole from you and then she lied about it.  I know, but come on. She's only 10. She's going to do way stupider stuff when she's older.  Guaranteed.  I don't care. She is my daughter and you will be punished.  Sorry, kid.  I wish I was your daughter.  Is that so? Well, guess what? You're sleeping here tonight because I don't feel like being around either one of you.  Great.  Yeah, great. There was a point where the doctors didn't know if you were going to make it and your mom got so scared and she made a promise to God that if you were okay that she\

In [ ]:
final_prompt = prompt.format(context = context_text , question = question)

In [ ]:
final_prompt

" \n     you are a helpful assistant.\n     Answer ONLY from the provided transcript context.\n     If the context is insufficient, just say you don't know.\n\n     you look like Madonna?  Thank you. I was actually trying to look like you.  Thank you.  Why would you give her makeup without talking to me first?  I didn't give her any makeup. Well, then one of you is lying.  She is.  She is.  Time to come clean, little lady. I ain't taking the hit on this one.  I took the makeup from me, mouse bag.  Oh, Missy, you are in a world of trouble.  Calm down. It's not that big a deal.  It is so a big deal. She stole from you and then she lied about it.  I know, but come on. She's only 10. She's going to do way stupider stuff when she's older.  Guaranteed.  I don't care. She is my daughter and you will be punished.  Sorry, kid.  I wish I was your daughter.  Is that so? Well, guess what? You're sleeping here tonight because I don't feel like being around either one of you.  Great.  Yeah, great. T

## Step 4 - Generation

In [ ]:
answer = llm.invoke(final_prompt)
print(answer)

content='Yes, Missy stole makeup.' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 1164, 'total_tokens': 1172, 'completion_time': 0.014575569, 'completion_tokens_details': None, 'prompt_time': 0.079450805, 'prompt_tokens_details': None, 'queue_time': 0.056379982, 'total_time': 0.094026374}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019e44eb-e2f9-75b1-89c9-7a68999da855-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 1164, 'output_tokens': 8, 'total_tokens': 1172}


## Building a Chain

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [ ]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [ ]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [ ]:
parallel_chain.invoke('Do missy stole makeup?')

{'context': "you look like Madonna?  Thank you. I was actually trying to look like you.  Thank you.  Why would you give her makeup without talking to me first?  I didn't give her any makeup. Well, then one of you is lying.  She is.  She is.  Time to come clean, little lady. I ain't taking the hit on this one.  I took the makeup from me, mouse bag.  Oh, Missy, you are in a world of trouble.  Calm down. It's not that big a deal.  It is so a big deal. She stole from you and then she lied about it.  I know, but come on. She's only 10. She's going to do way stupider stuff when she's older.  Guaranteed.  I don't care. She is my daughter and you will be punished.  Sorry, kid.  I wish I was your daughter.  Is that so? Well, guess what? You're sleeping here tonight because I don't feel like being around either one of you.  Great.  Yeah, great. There was a point where the doctors didn't know if you were going to make it and your mom got so scared and she made a promise to God that if you were ok

In [ ]:
parser = StrOutputParser()

In [ ]:
main_chain = parallel_chain | prompt | llm | parser

In [ ]:
main_chain.invoke('Do missy stole makeup?')

'Yes, Missy stole makeup.'